# Dataset B — Plain MLP v1: Train-only 5-fold CV (PMLP-01)

Notebook này chỉ triển khai **PMLP-01** trên artifact preprocessing 5 fold của partition Train. Mỗi fold khởi tạo model và optimizer mới, dự báo đúng fold-validation, rồi ghép OOF theo `record_id`.

Ranh giới bắt buộc:

- không đọc hoặc sử dụng external Validation/Test;
- không fit lại preprocessing trong notebook này;
- không threshold selection, full-Train refit, checkpoint hoặc final evaluation;
- configuration vẫn là `provisional_starting_configuration`, không phải project default/final;
- kết quả chỉ được ghi atomically sau khi toàn bộ năm fold và OOF gates đều pass.

Notebook được bàn giao ở trạng thái **chưa execute**. Chạy notebook là một task riêng cần được cho phép rõ ràng.

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import platform
import shutil
import tempfile
import time
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import yaml
from scipy.sparse import isspmatrix_csr, load_npz
from sklearn.metrics import roc_auc_score

from src.dataset_b.models.plain_mlp import PlainMLPClassifier
from src.utils.path import PROJECT_ROOT

CONFIG_PATH = PROJECT_ROOT / 'config' / 'dataset_b' / 'plain_mlp_v1.yaml'
CV_ARTIFACT_DIR = (
    PROJECT_ROOT / 'data' / 'preprocessing' / 'dataset_b_v1'
    / 'cv_neural_preprocessing_v1'
)
MANIFEST_PATH = CV_ARTIFACT_DIR / 'manifest.json'
RESULT_DIR = PROJECT_ROOT / 'artifacts' / 'dataset_b' / 'plain_mlp_v1'
WRAPPER_PATH = PROJECT_ROOT / 'src' / 'dataset_b' / 'models' / 'plain_mlp.py'
OUTPUT_NAMES = (
    'cv_results.json',
    'cv_oof_predictions.csv',
    'cv_training_history.json',
)

In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def require(condition: bool, message: str) -> None:
    if not condition:
        raise ValueError(message)


config = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))

require(config['model_version'] == 'plain_mlp_v1', 'Unexpected plain MLP model version.')
require(
    config['status'] == 'provisional_starting_configuration',
    'PMLP-01 requires the provisional starting configuration.',
)
require(config['is_project_default'] is False, 'PMLP-01 cannot use a project-default flag.')
require(config['is_final_configuration'] is False, 'PMLP-01 cannot use a final flag.')
require(config['selected_by_train_cv'] is False, 'PMLP-01 cannot pre-claim CV selection.')
require(config['environment']['device'] == 'cpu', 'plain_mlp_v1 is locked to CPU.')
require(config['environment']['torch_build'] == 'cpu', 'A CPU-only PyTorch build is required.')
require(not torch.cuda.is_available(), 'PMLP-01 must not silently switch to a GPU runtime.')
require(
    platform.python_version() == config['environment']['python_version'],
    'Python version differs from the locked neural environment.',
)
require(
    version('torch').split('+', maxsplit=1)[0] == str(config['environment']['torch_release']),
    'PyTorch release differs from the locked neural environment.',
)

require(manifest['source_partition'] == 'train', 'Neural CV artifact is not Train-only.')
require(manifest['fit_scope'] == 'fold_train_only', 'Unexpected preprocessing fit scope.')
require(manifest['use'] == 'cross_validation_only', 'Artifact is not marked CV-only.')
require(manifest['fold_count'] == 5, 'PMLP-01 requires exactly five folds.')
require(manifest['matrix_count'] == 10, 'PMLP-01 requires ten fold matrices.')
require(manifest['encoded_feature_count'] == 66, 'Expected the locked 66-column input width.')
require(manifest['matrix_format'] == 'csr', 'Expected CSR fold matrices.')
require(manifest['matrix_dtype'] == 'float32', 'Expected float32 fold matrices.')
require(
    manifest['preprocessing_version'] == config['contracts']['preprocessing_version'],
    'Preprocessing version differs between config and artifact.',
)
require(
    manifest['source_hashes']['plain_mlp_config']['sha256'] == sha256_file(CONFIG_PATH),
    'The PMLP config changed after preprocessing export.',
)

if RESULT_DIR.exists():
    raise FileExistsError(f'PMLP-01 refuses to overwrite an existing result directory: {RESULT_DIR}')

for relative_path, expected_hash in manifest['artifact_hashes'].items():
    artifact_path = CV_ARTIFACT_DIR / relative_path
    require(artifact_path.is_file(), f'Missing neural CV artifact: {relative_path}')
    require(sha256_file(artifact_path) == expected_hash, f'Hash mismatch: {relative_path}')

In [ ]:
fold_inputs: list[dict[str, object]] = []
fold_entries = sorted(manifest['folds'], key=lambda item: item['fold'])
require([entry['fold'] for entry in fold_entries] == [1, 2, 3, 4, 5], 'Invalid fold numbering.')
n_train = sum(int(entry['fold_validation_rows']) for entry in fold_entries)
require(n_train > 0, 'The Train partition is empty.')

oof_record_ids = np.empty(n_train, dtype=object)
oof_labels = np.full(n_train, -1, dtype=np.int8)
oof_fold = np.zeros(n_train, dtype=np.int8)
coverage = np.zeros(n_train, dtype=np.int8)

for entry in fold_entries:
    fold_number = int(entry['fold'])
    fold_dir = CV_ARTIFACT_DIR / str(entry['directory'])
    rows_path = fold_dir / 'rows.npz'
    with np.load(rows_path, allow_pickle=False) as rows:
        train_positions = rows['train_positions'].astype(np.int64, copy=False)
        validation_positions = rows['fold_validation_positions'].astype(np.int64, copy=False)
        train_record_ids = rows['train_record_ids'].astype(str, copy=False)
        validation_record_ids = rows['fold_validation_record_ids'].astype(str, copy=False)
        train_y = rows['train_y'].astype(np.int8, copy=False)
        validation_y = rows['fold_validation_y'].astype(np.int8, copy=False)

    require(len(train_positions) == int(entry['train_rows']), f'Fold {fold_number}: train row mismatch.')
    require(
        len(validation_positions) == int(entry['fold_validation_rows']),
        f'Fold {fold_number}: validation row mismatch.',
    )
    require(len(train_record_ids) == len(train_positions), f'Fold {fold_number}: train ID mismatch.')
    require(
        len(validation_record_ids) == len(validation_positions),
        f'Fold {fold_number}: validation ID mismatch.',
    )
    require(len(train_y) == len(train_positions), f'Fold {fold_number}: train label mismatch.')
    require(len(validation_y) == len(validation_positions), f'Fold {fold_number}: validation label mismatch.')
    require(set(np.unique(train_y).tolist()) == {0, 1}, f'Fold {fold_number}: train is not binary.')
    require(set(np.unique(validation_y).tolist()) == {0, 1}, f'Fold {fold_number}: validation is not binary.')
    require(
        np.intersect1d(train_positions, validation_positions).size == 0,
        f'Fold {fold_number}: train/validation overlap.',
    )
    require(
        np.array_equal(
            np.sort(np.concatenate((train_positions, validation_positions))),
            np.arange(n_train),
        ),
        f'Fold {fold_number}: positions do not partition Train.',
    )
    require(np.all(coverage[validation_positions] == 0), f'Fold {fold_number}: duplicate OOF positions.')

    coverage[validation_positions] = 1
    oof_record_ids[validation_positions] = validation_record_ids
    oof_labels[validation_positions] = validation_y
    oof_fold[validation_positions] = fold_number
    fold_inputs.append(
        {
            'fold': fold_number,
            'train_matrix_path': fold_dir / 'plain_mlp_train.npz',
            'validation_matrix_path': fold_dir / 'plain_mlp_fold_validation.npz',
            'train_positions': train_positions,
            'validation_positions': validation_positions,
            'train_y': train_y,
            'validation_y': validation_y,
        }
    )

require(np.all(coverage == 1), 'Fold-validation rows do not cover Train exactly once.')
require(np.all(oof_labels >= 0), 'Some OOF labels were not assigned.')
require(np.all(oof_fold > 0), 'Some OOF fold labels were not assigned.')
require(len(set(oof_record_ids.tolist())) == n_train, 'Train record_id values are not unique.')

In [ ]:
training_config = config['training']
architecture_config = config['architecture']
class_handling = config['class_handling']
require(training_config['loss'] == 'BCEWithLogitsLoss', 'Unexpected loss contract.')
require(training_config['optimizer'] == 'AdamW', 'Unexpected optimizer contract.')
require(training_config['early_stopping'] is False, 'PMLP-01 does not use early stopping.')
require(architecture_config['activation'] == 'ReLU', 'Unexpected activation contract.')
require(architecture_config['output_units'] == 1, 'Binary plain MLP requires one output logit.')
require(
    class_handling == {
        'class_weight': False,
        'sample_weight': False,
        'resampling': False,
    },
    'PMLP-01 does not permit class weighting, sample weighting, or resampling.',
)


def make_estimator() -> PlainMLPClassifier:
    return PlainMLPClassifier(
        hidden_layers=tuple(architecture_config['hidden_layers']),
        activation=architecture_config['activation'],
        learning_rate=float(training_config['learning_rate']),
        weight_decay=float(training_config['weight_decay']),
        dropout=float(training_config['dropout']),
        batch_size=int(training_config['batch_size']),
        epochs=int(training_config['epochs']),
        random_state=int(training_config['random_state']),
        device=config['environment']['device'],
        deterministic_algorithms=bool(training_config['deterministic_algorithms']),
    )

In [ ]:
oof_probability = np.full(n_train, np.nan, dtype=np.float64)
fold_results: list[dict[str, object]] = []
fold_histories: list[dict[str, object]] = []
fitted_estimators: list[PlainMLPClassifier] = []
cv_started = time.perf_counter()

for fold_input in fold_inputs:
    fold_number = int(fold_input['fold'])
    X_train = load_npz(fold_input['train_matrix_path'])
    X_validation = load_npz(fold_input['validation_matrix_path'])
    train_y = fold_input['train_y']
    validation_y = fold_input['validation_y']
    validation_positions = fold_input['validation_positions']

    require(isspmatrix_csr(X_train), f'Fold {fold_number}: train matrix is not CSR.')
    require(isspmatrix_csr(X_validation), f'Fold {fold_number}: validation matrix is not CSR.')
    require(X_train.dtype == np.float32, f'Fold {fold_number}: train matrix is not float32.')
    require(X_validation.dtype == np.float32, f'Fold {fold_number}: validation matrix is not float32.')
    require(X_train.shape == (len(train_y), 66), f'Fold {fold_number}: train shape mismatch.')
    require(
        X_validation.shape == (len(validation_y), 66),
        f'Fold {fold_number}: validation shape mismatch.',
    )
    require(np.isfinite(X_train.data).all(), f'Fold {fold_number}: non-finite train value.')
    require(np.isfinite(X_validation.data).all(), f'Fold {fold_number}: non-finite validation value.')

    estimator = make_estimator()
    fold_started = time.perf_counter()
    estimator.fit(X_train, train_y)
    probability = estimator.predict_proba(X_validation)[:, 1]
    fold_seconds = time.perf_counter() - fold_started

    require(probability.shape == (len(validation_y),), f'Fold {fold_number}: probability shape mismatch.')
    require(np.isfinite(probability).all(), f'Fold {fold_number}: non-finite probabilities.')
    require(np.all((probability >= 0.0) & (probability <= 1.0)), f'Fold {fold_number}: invalid probability.')
    require(
        all(
            estimator.model_ is not previous.model_
            and estimator.optimizer_ is not previous.optimizer_
            for previous in fitted_estimators
        ),
        f'Fold {fold_number}: fitted model or optimizer state was reused.',
    )

    fold_auc = float(roc_auc_score(validation_y, probability))
    oof_probability[validation_positions] = probability
    fold_results.append(
        {
            'fold': fold_number,
            'train_rows': int(X_train.shape[0]),
            'fold_validation_rows': int(X_validation.shape[0]),
            'roc_auc': fold_auc,
            'runtime_seconds': float(fold_seconds),
        }
    )
    fold_histories.append(
        {
            'fold': fold_number,
            'epoch_count': len(estimator.training_history_),
            'training_loss': [float(value) for value in estimator.training_history_],
        }
    )
    fitted_estimators.append(estimator)

cv_runtime_seconds = time.perf_counter() - cv_started

In [ ]:
require(len(fold_results) == 5, 'PMLP-01 did not complete exactly five folds.')
require(np.isfinite(oof_probability).all(), 'OOF probabilities do not cover all Train rows.')
require(
    all(item['epoch_count'] == int(training_config['epochs']) for item in fold_histories),
    'A fold did not complete the locked epoch count.',
)

fold_auc_values = np.asarray([item['roc_auc'] for item in fold_results], dtype=np.float64)
oof_predictions = pd.DataFrame(
    {
        'record_id': oof_record_ids.astype(str),
        'fold': oof_fold,
        'y_approved': oof_labels,
        'p_approved': oof_probability,
    }
)
require(len(oof_predictions) == n_train, 'OOF artifact row count differs from Train.')
require(oof_predictions['record_id'].is_unique, 'OOF artifact contains duplicate record_id values.')
require(
    sorted(oof_predictions['fold'].unique().tolist()) == [1, 2, 3, 4, 5],
    'OOF artifact does not contain all five folds.',
)

source_sha256 = {
    'plain_mlp_config': {
        'path': CONFIG_PATH.relative_to(PROJECT_ROOT).as_posix(),
        'sha256': sha256_file(CONFIG_PATH),
    },
    'neural_preprocessing_manifest': {
        'path': MANIFEST_PATH.relative_to(PROJECT_ROOT).as_posix(),
        'sha256': sha256_file(MANIFEST_PATH),
    },
    'plain_mlp_wrapper': {
        'path': WRAPPER_PATH.relative_to(PROJECT_ROOT).as_posix(),
        'sha256': sha256_file(WRAPPER_PATH),
    },
}

history_payload = {
    'schema_version': 1,
    'dataset': 'dataset_b',
    'model_version': config['model_version'],
    'stage': 'train_only_cross_validation',
    'source_partition': 'train',
    'external_validation_used': False,
    'test_used': False,
    'folds': fold_histories,
}
results_payload = {
    'schema_version': 1,
    'dataset': 'dataset_b',
    'model': 'plain_mlp',
    'model_version': config['model_version'],
    'stage': 'train_only_cross_validation',
    'status': 'complete',
    'configuration_status': config['status'],
    'is_project_default': False,
    'is_final_configuration': False,
    'selected_by_train_cv': False,
    'feature_version': manifest['feature_version'],
    'split_version': manifest['split_version'],
    'preprocessing_version': manifest['preprocessing_version'],
    'source_partition': 'train',
    'fit_scope': 'fold_train_only',
    'external_validation_used': False,
    'test_used': False,
    'fold_count': 5,
    'train_rows': int(n_train),
    'encoded_feature_count': 66,
    'primary_metric': 'roc_auc',
    'roc_auc_std_ddof': 0,
    'mean_roc_auc': float(fold_auc_values.mean()),
    'std_roc_auc': float(fold_auc_values.std(ddof=0)),
    'oof_roc_auc': float(roc_auc_score(oof_labels, oof_probability)),
    'runtime_seconds': float(cv_runtime_seconds),
    'estimated_optimizer_steps': int(
        sum(math.ceil(item['train_rows'] / int(training_config['batch_size'])) for item in fold_results)
        * int(training_config['epochs'])
    ),
    'parameters': {
        'architecture': architecture_config,
        'training': training_config,
        'class_handling': class_handling,
    },
    'folds': fold_results,
    'source_sha256': source_sha256,
    'library_versions': {
        'python': platform.python_version(),
        'numpy': version('numpy'),
        'pandas': version('pandas'),
        'scikit-learn': version('scikit-learn'),
        'scipy': version('scipy'),
        'torch': version('torch'),
    },
}

In [ ]:
RESULT_DIR.parent.mkdir(parents=True, exist_ok=True)
staging_dir = Path(
    tempfile.mkdtemp(prefix=f'.{RESULT_DIR.name}.pmlp01-staging-', dir=RESULT_DIR.parent)
)
try:
    oof_path = staging_dir / 'cv_oof_predictions.csv'
    history_path = staging_dir / 'cv_training_history.json'
    results_path = staging_dir / 'cv_results.json'

    oof_predictions.to_csv(oof_path, index=False)
    history_path.write_text(
        json.dumps(history_payload, ensure_ascii=False, indent=2),
        encoding='utf-8',
    )
    results_payload['output_sha256'] = {
        oof_path.name: sha256_file(oof_path),
        history_path.name: sha256_file(history_path),
    }
    results_path.write_text(
        json.dumps(results_payload, ensure_ascii=False, indent=2),
        encoding='utf-8',
    )
    require(
        sorted(path.name for path in staging_dir.iterdir()) == sorted(OUTPUT_NAMES),
        'PMLP-01 staging directory contains an unexpected file set.',
    )
    staging_dir.replace(RESULT_DIR)
except Exception:
    if staging_dir.exists():
        shutil.rmtree(staging_dir)
    raise

{
    'result_directory': RESULT_DIR.relative_to(PROJECT_ROOT).as_posix(),
    'fold_roc_auc': [item['roc_auc'] for item in fold_results],
    'mean_roc_auc': results_payload['mean_roc_auc'],
    'std_roc_auc': results_payload['std_roc_auc'],
    'oof_rows': len(oof_predictions),
    'runtime_seconds': results_payload['runtime_seconds'],
    'configuration_status': results_payload['configuration_status'],
}